# 🧠 Hallucination test · Entity-recognition features in Qwen3.6-27B

Replication-style test of [Ferrando et al. ICLR 2025 ("Do I Know This Entity?")](https://arxiv.org/abs/2411.14257) on our paper-grade SAEs at L11 / L31 / L55.

**Hypothesis**: SAE features encode an "I know this entity" signal that distinguishes Wikipedia-grade real entities from plausibly-named-but-non-existent ones. If the signal is strong, ablating it should reduce confabulation on fake entities (= the standard SAE-for-hallucination story).

**Pipeline**:
1. Build a dataset of 200 known entities + 200 synthetic unknowns (name-like strings with zero Wikipedia presence)
2. Forward pass with `"The famous person {entity} was"` prompts, capture SAE activations at the entity's last token in L11/L31/L55
3. Per-feature separation: `(mean z_known) − (mean z_unknown)` — Cohen's d-like score
4. Train sklearn LogisticRegression on top-K features → known/unknown AUROC per layer
5. **Stage-gate decision**:
   - **AUROC ≥ 0.65** → entity-recognition signal real → ship + writeup
   - **AUROC 0.55-0.65** → weak signal → document, don't claim victory
   - **AUROC < 0.55** → honest negative → writeup as 'first 27B test, didn't transfer from Gemma'

**Reference numbers** (Ferrando 2024 on Gemma-2-2B): AUROC ≈ 0.732 at best layer (L9 of Gemma-2-2B). Our 27B model is much larger and reasoning-tuned — could go either way.

**Cost**: ~$5 GPU on Colab RTX 6000 Pro · ~25 min wall time.

Output: separation scores per feature × layer + AUROC summary, both uploaded to the SAE HF repo as `hallucination_v0_0_1.json`.

In [ ]:
!pip install -q -U transformers accelerate safetensors huggingface_hub scikit-learn matplotlib tqdm
import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 1. Config + load base model + 3 SAEs (frozen)

In [ ]:
HF_SAE_REPO   = 'caiovicentino1/qwen36-27b-sae-papergrade'
HF_BASE_MODEL = 'Qwen/Qwen3.6-27B'
LAYERS        = [11, 31, 55]
D_MODEL       = 5120
D_SAE         = 65_536
K             = 128

PROMPT_TEMPLATE = 'The famous person {entity} was'   # capture at last token of {entity}
TOP_FEATURES_FOR_PROBE = 50      # top features by |separation| used as LogReg inputs
TRAIN_FRAC = 0.7
SEED = 0

import os, math, json, time, random
random.seed(SEED); torch.manual_seed(SEED)

from huggingface_hub import login, hf_hub_download
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

from transformers import AutoTokenizer, AutoModelForImageTextToText
from safetensors.torch import load_file
import torch.nn.functional as F

device = 'cuda'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map='cuda',
    trust_remote_code=True,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

class TopKSAE(torch.nn.Module):
    def __init__(self, sd, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16), requires_grad=False)
        self.b_enc = torch.nn.Parameter(sd['b_enc'].to(torch.bfloat16), requires_grad=False)
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16), requires_grad=False)
        self.b_dec = torch.nn.Parameter(sd['b_dec'].to(torch.bfloat16), requires_grad=False)
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, vals)
        return z

saes = {}
for layer in LAYERS:
    path = hf_hub_download(HF_SAE_REPO, f'sae_L{layer}_latest.safetensors')
    saes[layer] = TopKSAE(load_file(path), K).to(device).eval()
    print(f'  ✓ SAE L{layer}')

layer_mods = {layer: model.model.language_model.layers[layer] for layer in LAYERS}
print(f'\nready · vram free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## 2. Build entity datasets

**Known**: 200 unambiguously famous people (actors, scientists, athletes, historical figures) — names that any encyclopedic LLM should recognize.

**Unknown**: 200 synthetic names with the **same surface structure** (Slavic / Italianate / Latinate roots that look real) but **zero Wikipedia presence**. Generated procedurally from name-part pools designed to not match real combinations.

Both lists are hardcoded so this notebook reproduces deterministically. They're inspectable below — please flag any duplicates or accidentally-real names.

In [ ]:
# Unambiguously famous people across fields — Wikipedia-grade entities.
# Hand-curated; trim to N_KNOWN below to keep counts symmetric with unknowns.
KNOWN_ENTITIES = [
    # Actors / Entertainers
    'Tom Hanks', 'Meryl Streep', 'Denzel Washington', 'Robert De Niro', 'Al Pacino',
    'Leonardo DiCaprio', 'Brad Pitt', 'Angelina Jolie', 'Scarlett Johansson', 'Morgan Freeman',
    'Jennifer Lawrence', 'Will Smith', 'Cate Blanchett', 'Anthony Hopkins', 'Judi Dench',
    'Daniel Day-Lewis', 'Helen Mirren', 'Christian Bale', 'Viola Davis', 'Joaquin Phoenix',
    'Frances McDormand', 'Tom Hardy', 'Kate Winslet', 'Dustin Hoffman', 'Jack Nicholson',
    # Musicians
    'Bob Dylan', 'Paul McCartney', 'Mick Jagger', 'David Bowie', 'Madonna',
    'Beyoncé', 'Taylor Swift', 'Kanye West', 'Drake', 'Rihanna',
    'Adele', 'Bruno Mars', 'Lady Gaga', 'Justin Bieber', 'Ed Sheeran',
    'Freddie Mercury', 'John Lennon', 'Elvis Presley', 'Michael Jackson', 'Prince',
    'Stevie Wonder', 'Aretha Franklin', 'Whitney Houston', 'Tupac Shakur', 'The Notorious B.I.G.',
    # Athletes
    'Lionel Messi', 'Cristiano Ronaldo', 'LeBron James', 'Michael Jordan', 'Serena Williams',
    'Roger Federer', 'Rafael Nadal', 'Tom Brady', 'Tiger Woods', 'Usain Bolt',
    'Pelé', 'Diego Maradona', 'Kobe Bryant', 'Magic Johnson', 'Larry Bird',
    'Mike Tyson', 'Muhammad Ali', 'Floyd Mayweather', 'Lewis Hamilton', 'Ayrton Senna',
    'Michael Schumacher', 'Simone Biles', 'Michael Phelps', 'Wayne Gretzky', 'Cristina Aguilera',
    # Scientists / Inventors
    'Albert Einstein', 'Isaac Newton', 'Marie Curie', 'Charles Darwin', 'Galileo Galilei',
    'Stephen Hawking', 'Nikola Tesla', 'Thomas Edison', 'Alan Turing', 'Richard Feynman',
    'Niels Bohr', 'Werner Heisenberg', 'Erwin Schrödinger', 'Max Planck', 'Enrico Fermi',
    'James Clerk Maxwell', 'Michael Faraday', 'Louis Pasteur', 'Jonas Salk', 'Linus Pauling',
    'Carl Sagan', 'Neil deGrasse Tyson', 'Watson and Crick', 'Rosalind Franklin', 'Ada Lovelace',
    # Historical figures / Politicians
    'Abraham Lincoln', 'George Washington', 'Theodore Roosevelt', 'Franklin Roosevelt', 'John F. Kennedy',
    'Barack Obama', 'Winston Churchill', 'Margaret Thatcher', 'Mahatma Gandhi', 'Nelson Mandela',
    'Martin Luther King Jr.', 'Malcolm X', 'Mother Teresa', 'Pope Francis', 'Pope John Paul II',
    'Napoleon Bonaparte', 'Julius Caesar', 'Alexander the Great', 'Cleopatra', 'Joan of Arc',
    'Queen Elizabeth II', 'King Henry VIII', 'Queen Victoria', 'Catherine the Great', 'Vladimir Lenin',
    # Writers / Philosophers
    'William Shakespeare', 'Jane Austen', 'Charles Dickens', 'Mark Twain', 'Ernest Hemingway',
    'F. Scott Fitzgerald', 'Virginia Woolf', 'James Joyce', 'George Orwell', 'J.R.R. Tolkien',
    'J.K. Rowling', 'Stephen King', 'Toni Morrison', 'Gabriel García Márquez', 'Jorge Luis Borges',
    'Plato', 'Aristotle', 'Socrates', 'Friedrich Nietzsche', 'Immanuel Kant',
    'Karl Marx', 'Sigmund Freud', 'Carl Jung', 'Jean-Paul Sartre', 'Simone de Beauvoir',
    # Tech / Business
    'Steve Jobs', 'Bill Gates', 'Mark Zuckerberg', 'Elon Musk', 'Jeff Bezos',
    'Larry Page', 'Sergey Brin', 'Tim Cook', 'Sundar Pichai', 'Satya Nadella',
    'Sam Altman', 'Dario Amodei', 'Yann LeCun', 'Geoffrey Hinton', 'Andrej Karpathy',
    'Demis Hassabis', 'Ilya Sutskever', 'Andrew Ng', 'Fei-Fei Li', 'Lex Fridman',
    'Warren Buffett', 'Charlie Munger', 'George Soros', 'Ray Dalio', 'Peter Thiel',
    # Artists
    'Pablo Picasso', 'Vincent van Gogh', 'Leonardo da Vinci', 'Michelangelo', 'Claude Monet',
    'Salvador Dalí', 'Frida Kahlo', 'Andy Warhol', 'Jackson Pollock', 'Georgia O’Keeffe',
    # Directors
    'Steven Spielberg', 'Martin Scorsese', 'Quentin Tarantino', 'Christopher Nolan', 'Stanley Kubrick',
    'Alfred Hitchcock', 'Akira Kurosawa', 'Federico Fellini', 'Ingmar Bergman', 'David Lynch',
    # Comedians / TV
    'Charlie Chaplin', 'Jerry Seinfeld', 'Dave Chappelle', 'Robin Williams', 'Eddie Murphy',
    'Oprah Winfrey', 'Jon Stewart', 'Stephen Colbert', 'David Letterman', 'Jay Leno',
]

# Trim to N_KNOWN to handle accidental over-curation. Keeps known/unknown symmetric.
N_KNOWN = 200
KNOWN_ENTITIES = KNOWN_ENTITIES[:N_KNOWN]
assert len(KNOWN_ENTITIES) == N_KNOWN, f'expected {N_KNOWN} known, got {len(KNOWN_ENTITIES)}'
print(f'  known entities: {len(KNOWN_ENTITIES)}')

In [ ]:
# 200 synthetic unknowns: plausibly-named but with zero Wikipedia presence.
# Strategy: hand-design name parts from Slavic / Latinate / Italianate roots that
# are surface-plausible (English speakers wouldn't immediately go "that's fake")
# but specific combinations don't appear in any major encyclopedia.
FIRSTS = [
    'Vlasik','Krenadia','Tomalin','Oranthia','Brastov','Nelvian','Cordrick','Velinia',
    'Pradon','Sevirin','Mirkali','Dorvain','Kestrel','Velmirov','Astrid-Vex','Brennan-Os',
    'Ostrigan','Calverin','Drestov','Ulyanov-Brek','Marvinec','Tagrin','Ozalin','Versan',
    'Welkar','Petric-Os','Belmondrov','Astravel','Quinten-Var','Renjik','Polvashin','Halverin',
    'Bostran','Crelvi','Jorthal','Remalin','Ozurik','Velkrov','Pelvad','Trinov',
]
LASTS = [
    'Korpel','Brovner','Veshenko','Vexbridge','Marinescu','Drobenov','Kalsten','Volotov',
    'Pendrigan','Brastovic','Korbenev','Sondrik','Mirovich','Trenavod','Volenov','Pradetski',
    'Karlovich-Sten','Drovenko','Brakovic','Pestrini','Grevolic','Tarminev','Kostrini','Velnikov',
    'Ralvenov','Drevich','Kazinev','Petrolic','Stranev','Kolzic','Belnov','Vrastik',
    'Dorinov','Krasten','Volsten','Mirnec','Drasenov','Kolbric','Penzic','Volnesti',
]
rng = random.Random(SEED)
raw_pairs = [(f, l) for f in FIRSTS for l in LASTS]
rng.shuffle(raw_pairs)
UNKNOWN_ENTITIES = []
seen_lower = {e.lower() for e in KNOWN_ENTITIES}
for first, last in raw_pairs:
    name = f'{first} {last}'
    if name.lower() in seen_lower:
        continue
    UNKNOWN_ENTITIES.append(name)
    if len(UNKNOWN_ENTITIES) == 200:
        break
assert len(UNKNOWN_ENTITIES) == 200
print(f'  unknown entities: {len(UNKNOWN_ENTITIES)}')
print('  sample known:    ', KNOWN_ENTITIES[::40])
print('  sample unknown:  ', UNKNOWN_ENTITIES[::40])

## 3. Forward pass + capture activations at the entity's last token

For each prompt `"The famous person {entity} was"`, find the position of the entity's last token (right before `" was"`) and capture the residual at L11/L31/L55.

In [ ]:
from tqdm.auto import tqdm

_captured = {}
def capture_hook(key):
    def h(mod, inp, out):
        _captured[key] = out[0] if isinstance(out, tuple) else out
        return out
    return h

def find_entity_last_pos(prompt: str, entity: str) -> int:
    """Return token position (0-indexed) of the last token of `entity` inside the prompt.
    Falls back to position right before the ' was' suffix."""
    full_ids = tok(prompt, return_tensors='pt')['input_ids'][0].tolist()
    suffix = ' was'
    suffix_ids = tok(suffix, add_special_tokens=False)['input_ids']
    # Find suffix in full_ids, return position right before suffix start
    n = len(full_ids); m = len(suffix_ids)
    for i in range(n - m, -1, -1):
        if full_ids[i:i+m] == suffix_ids:
            return i - 1
    # Fallback: last token
    return n - 1

def encode_z_at_position(entity_list, label):
    """Run forward + capture activations at entity's last token. Returns z dict {layer: (N, D_SAE) numpy}."""
    import numpy as np
    z_per_layer = {layer: [] for layer in LAYERS}
    hs = []
    for layer in LAYERS:
        hs.append(layer_mods[layer].register_forward_hook(capture_hook(layer)))
    with torch.no_grad():
        for entity in tqdm(entity_list, desc=label):
            prompt = PROMPT_TEMPLATE.format(entity=entity)
            ids = tok(prompt, return_tensors='pt')['input_ids'].to(device)
            pos = find_entity_last_pos(prompt, entity)
            _ = model(ids)
            for layer in LAYERS:
                resid = _captured[layer][0, pos].to(torch.bfloat16)   # (D_MODEL,)
                z = saes[layer].encode(resid.unsqueeze(0))[0].float().cpu().numpy()
                z_per_layer[layer].append(z)
    for h in hs:
        h.remove()
    return {layer: np.stack(z_per_layer[layer], axis=0) for layer in LAYERS}

Z_known   = encode_z_at_position(KNOWN_ENTITIES,   'known')
Z_unknown = encode_z_at_position(UNKNOWN_ENTITIES, 'unknown')

for layer in LAYERS:
    print(f'  L{layer}: Z_known={Z_known[layer].shape}, Z_unknown={Z_unknown[layer].shape}')

## 4. Per-feature separation score + top-K extraction

For each feature `f` we compute Cohen's-d-style separation:

```
sep(f) = (mean(Z_known[:, f]) - mean(Z_unknown[:, f])) / (pooled_std + 1e-9)
```

Positive `sep` ⇒ feature fires more on known entities ("I know this" feature). Negative ⇒ fires more on unknowns (rare; would be a "genuinely strange" feature).

In [ ]:
import numpy as np

separation = {}
top_features = {}
for layer in LAYERS:
    zk = Z_known[layer]        # (N_known, D_SAE)
    zu = Z_unknown[layer]      # (N_unknown, D_SAE)
    mu_k, mu_u = zk.mean(axis=0), zu.mean(axis=0)
    sd_k, sd_u = zk.std(axis=0) + 1e-9, zu.std(axis=0) + 1e-9
    pooled = np.sqrt((sd_k**2 + sd_u**2) / 2)
    sep = (mu_k - mu_u) / (pooled + 1e-9)
    separation[layer] = sep
    top_idx = np.argsort(-np.abs(sep))[:TOP_FEATURES_FOR_PROBE]
    top_features[layer] = top_idx
    print(f'  L{layer}: |sep| max={np.abs(sep).max():.3f} top-50 mean |sep|={np.abs(sep[top_idx]).mean():.3f}')
    # Show the top 5 by |sep|
    print(f'    top 5 by |sep| (positive=known-firing, negative=unknown-firing):')
    for i in np.argsort(-np.abs(sep))[:5]:
        sign = '+' if sep[i] > 0 else '-'
        print(f'      {sign} L{layer}/f{i:5d}  sep={sep[i]:+.3f}  '
              f'mu_known={mu_k[i]:.3f}  mu_unk={mu_u[i]:.3f}')

## 5. Probe AUROC — known vs unknown classification

Logistic regression on the top-50 features (by |separation|) with 70/30 train/test split. AUROC measures how well SAE features distinguish known from unknown entities at each layer.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

auroc = {}
best_layer = None
best_auroc = -1.0
for layer in LAYERS:
    feats = top_features[layer]
    X_known = Z_known[layer][:, feats]
    X_unk   = Z_unknown[layer][:, feats]
    X = np.concatenate([X_known, X_unk], axis=0)
    y = np.concatenate([np.ones(len(X_known)), np.zeros(len(X_unk))])
    # Shuffle + split
    idx = np.arange(len(y))
    np.random.seed(SEED); np.random.shuffle(idx)
    n_train = int(TRAIN_FRAC * len(y))
    tr, te = idx[:n_train], idx[n_train:]
    clf = LogisticRegression(max_iter=2000, C=1.0).fit(X[tr], y[tr])
    auc = float(roc_auc_score(y[te], clf.predict_proba(X[te])[:, 1]))
    auroc[layer] = auc
    print(f'  L{layer}: AUROC = {auc:.4f}')
    if auc > best_auroc:
        best_auroc, best_layer = auc, layer

print(f'\n  best layer: L{best_layer}  AUROC={best_auroc:.4f}')

## 6. Stage-gate decision

Following the protocol from [Ferrando 2024](https://arxiv.org/abs/2411.14257), which reported AUROC ≈ 0.732 on Gemma-2-2B as the meaningful threshold. Our gate:

- **AUROC ≥ 0.65** → entity-recognition signal is real → ship + writeup.
- **AUROC 0.55–0.65** → weak signal → publish honestly with caveat.
- **AUROC < 0.55** → honest negative → first 27B test, didn't transfer.

In [ ]:
if best_auroc >= 0.65:
    verdict = 'STRONG · ship + writeup'
    color = '\u2705'   # green check
elif best_auroc >= 0.55:
    verdict = 'WEAK · document with caveat'
    color = '\u26a0\ufe0f'   # warning
else:
    verdict = 'NEGATIVE · honest writeup as first 27B replication failure'
    color = '\u274c'   # red x

print(f'\n{"="*60}\n{color}  STAGE-GATE: {verdict}\n{"="*60}')
print(f'  best layer: L{best_layer}  AUROC={best_auroc:.4f}')
print(f'  reference (Ferrando 2024 on Gemma-2-2B): 0.732 at L9')
print(f'\n  per-layer:')
for layer in LAYERS:
    print(f'    L{layer}: {auroc[layer]:.4f}')

## 7. Inspect the top entity-recognition feature

On the best layer, look at the single feature with strongest separation. What does its activation distribution look like? Which 3 known entities and 3 unknown entities does it fire most strongly on? This is qualitative sanity for whether it's really "I know this" or some artifact.

In [ ]:
import matplotlib.pyplot as plt

layer = best_layer
best_feat = int(np.argsort(-np.abs(separation[layer]))[0])
sep_val = separation[layer][best_feat]
zk = Z_known[layer][:, best_feat]
zu = Z_unknown[layer][:, best_feat]

print(f'Best feature on L{layer}: f{best_feat}  sep={sep_val:+.3f}')
print(f'  mean(known)   = {zk.mean():.3f}')
print(f'  mean(unknown) = {zu.mean():.3f}')
print(f'  std(known)    = {zk.std():.3f}')
print(f'  std(unknown)  = {zu.std():.3f}')

print(f'\nTop 5 KNOWN entities by activation:')
for i in np.argsort(-zk)[:5]:
    print(f'  {zk[i]:.3f}  {KNOWN_ENTITIES[i]}')
print(f'\nTop 5 UNKNOWN entities by activation:')
for i in np.argsort(-zu)[:5]:
    print(f'  {zu[i]:.3f}  {UNKNOWN_ENTITIES[i]}')

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=140)
ax.hist(zk, bins=30, alpha=0.55, color='#16a34a', label=f'known (n={len(zk)})')
ax.hist(zu, bins=30, alpha=0.55, color='#f97316', label=f'unknown (n={len(zu)})')
ax.axvline(zk.mean(), color='#16a34a', linestyle='--', linewidth=1)
ax.axvline(zu.mean(), color='#f97316', linestyle='--', linewidth=1)
ax.set_xlabel('activation')
ax.set_ylabel('count')
ax.set_title(f'L{layer}/f{best_feat}  ·  sep={sep_val:+.3f}  ·  AUROC@L{layer}={auroc[layer]:.3f}',
             fontsize=12, fontweight='bold')
ax.legend(frameon=False)
for s in ('top','right'): ax.spines[s].set_visible(False)
ax.grid(axis='y', color='#f3f4f6', linewidth=1)
plt.tight_layout()
chart_path = f'/tmp/hallucination_top_feature_L{layer}.png'
plt.savefig(chart_path, dpi=140, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\nsaved chart: {chart_path}')

## 8. Save artifacts to HF SAE repo

JSON with separation per feature × layer + AUROC summary + best-feature info. Lives at `hallucination_v0_0_1.json` next to the SAEs and gets cited from the writeup.

In [ ]:
from huggingface_hub import HfApi
from datetime import datetime, timezone

results = {
    'version':        'v0.0.1',
    'model':          HF_BASE_MODEL,
    'sae_repo':       HF_SAE_REPO,
    'method':         'Ferrando 2024 entity-recognition replication',
    'arxiv':          'arxiv:2411.14257',
    'prompt_template': PROMPT_TEMPLATE,
    'n_known':        len(KNOWN_ENTITIES),
    'n_unknown':      len(UNKNOWN_ENTITIES),
    'top_features_for_probe': TOP_FEATURES_FOR_PROBE,
    'train_frac':     TRAIN_FRAC,
    'auroc_per_layer': {str(l): auroc[l] for l in LAYERS},
    'best_layer':     int(best_layer),
    'best_auroc':     float(best_auroc),
    'verdict':        verdict,
    'top_separation_features': {
        str(layer): [
            {
                'feature': int(i),
                'separation': float(separation[layer][i]),
                'mean_known': float(Z_known[layer][:, i].mean()),
                'mean_unknown': float(Z_unknown[layer][:, i].mean()),
            }
            for i in np.argsort(-np.abs(separation[layer]))[:20]
        ]
        for layer in LAYERS
    },
    'reference': {
        'ferrando_2024_gemma2_2b_auroc': 0.732,
        'gate_threshold_strong': 0.65,
        'gate_threshold_weak':   0.55,
    },
    'timestamp':      datetime.now(timezone.utc).isoformat(),
}

out_path = '/tmp/hallucination_v0_0_1.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

api = HfApi()
api.upload_file(
    path_or_fileobj=out_path,
    path_in_repo='hallucination_v0_0_1.json',
    repo_id=HF_SAE_REPO,
    commit_message=f'Hallucination test v0.0.1 — {verdict.split(chr(183))[0].strip()} (best AUROC={best_auroc:.3f} @ L{best_layer})',
)
api.upload_file(
    path_or_fileobj=chart_path,
    path_in_repo='charts/hallucination_top_feature.png',
    repo_id=HF_SAE_REPO,
    commit_message='Hallucination v0.0.1 chart — best entity-recognition feature',
)
print(f'\n✓ uploaded → https://huggingface.co/{HF_SAE_REPO}')
print(f'\nNext: write up the result. Verdict above.')